In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t057/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task data
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is for a significant increase in response time for the operation `cart::POST /oteldemo.CartService/AddItem`. The alert time window is from `2026-04-04 22:37:09.273091` to `2026-04-04 22:43:03` (UTC+8). The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('', {'start': '2026-04-04T22:37:09.273091+08:00', 'end': '2026-04-04T22:43:03+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">cart响应时间突增告警<alert_event event_id="62ea6e8a5abb9b1a2ade04406f02d789" trans_id="9j2a8qucut0bt1pk31v0nuaef1" rule_id="e8b430fd-f3db-4f75-bf3b-11ed13c00b33" rule_name="cart响应时间突增告警" /></code>\n\n帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t057/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated components
kpi_components = metrics_data[['entity_set', 'entity_name', 'metric']].drop_duplicates()

kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs across multiple components. These include node-level metrics (e.g., `node_ready_status`, `node_memory_usage_rate`, `node_disk_usage_rate`, `node_pod_running_count`) and application-level metrics (e.g., `error_rate` for specific instances like `inventory-794d9f8f85-hhjtj` and `frontend-proxy-67d64dc4f8-8c99b`). There are a total of 298 unique combinations of entity sets, entity names, and metrics. For a more detailed view, additional rows can be displayed.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                      entity_name                  metric
0          k8s.node                                        node_ready_status
2040       k8s.node                                    node_not_ready_status
4080       k8s.node                                     node_disk_usage_rate
4216       k8s.node                                   node_memory_usage_rate
7208       k8s.node                                   node_pod_running_count
...             ...                              ...                     ...
86019  apm.instance       inventory-794d9f8f85-hhjtj              error_rate
86023  apm.instance  frontend-proxy-67d64dc4f8-8c99b              error_rate
86044  apm.instance  frontend-proxy-67d64dc4f8-4ptg6              error_rate
86057  apm.instance        frontend-6df7d557f5-nz7g4              error_rate
86070  apm.instance        inventory-887d8db89-7v4s5              error_rate

[298 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter metrics data to include only service-level KPIs (entity_set = 'apm.service.legacy')
service_metrics = metrics_data[metrics_data['entity_set'] == 'apm.service.legacy']

# Ensure 'value' column is numeric for aggregation
service_metrics['value'] = pd.to_numeric(service_metrics['value'], errors='coerce')

# Calculate global thresholds (e.g., P95) for each 'service-KPI' combination
service_kpi_thresholds = service_metrics.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
service_kpi_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

service_kpi_thresholds

```
Out[3]:
```
The global P95 thresholds for service-level KPIs have been calculated for each service-KPI combination. For example, the `accounting` service has a P95 threshold of `0.0077` for `cpu`, `0.0352` for `latency`, and `1.064e+08` for `mem`. Similarly, other services like `ad`, `recommendation`, and `shipping` have their respective thresholds for KPIs such as `cpu`, `latency`, `mem`, and `workload`. There are a total of 73 service-KPI combinations in the results.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name    metric  p95_threshold
0       accounting       cpu   7.731293e-03
1       accounting   latency   3.518571e-02
2       accounting       mem   1.064113e+08
3       accounting  workload   6.191000e+03
4               ad       cpu   1.440792e-03
..             ...       ...            ...
68  recommendation  workload   1.762500e+02
69        shipping       cpu   1.392222e-02
70        shipping   latency   1.450878e-03
71        shipping       mem   3.261099e+06
72        shipping  workload   1.238775e+04

[73 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Convert alert time window to microseconds for filtering
timezone = pytz.timezone('Asia/Shanghai')
alert_start = int(datetime.strptime('2026-04-04T22:37:09.273091+08:00', '%Y-%m-%dT%H:%M:%S.%f%z').timestamp() * 1_000_000)
alert_end = int(datetime.strptime('2026-04-04T22:43:03+08:00', '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter service-level metrics data within the alert time window
service_metrics_filtered = service_metrics[
    (service_metrics['time'] >= alert_start) & (service_metrics['time'] <= alert_end)
]

# Merge filtered metrics with P95 thresholds to identify anomalies
anomalies = service_metrics_filtered.merge(
    service_kpi_thresholds,
    on=['entity_name', 'metric'],
    how='inner'
)

# Identify anomalies where the KPI value exceeds the P95 threshold
anomalies = anomalies[anomalies['value'] > anomalies['p95_threshold']]

# Report anomalies with service name, KPI, and timestamps
anomalies_report = anomalies[['entity_name', 'metric', 'time', 'value', 'p95_threshold']]

anomalies_report

```
Out[4]:
```
During the alert time window, 141 anomalies were identified where KPI values exceeded their respective P95 thresholds. These anomalies span multiple services and KPIs. For example, the `ad` service experienced anomalies in `workload`, the `inventory` service in `mem`, the `cart` service in `mem`, and the `frontend-proxy` service in `latency`. Each anomaly includes the service name, KPI, timestamp, and the value exceeding the threshold. For a detailed view, additional rows can be displayed.

The original code execution output of IPython Kernel is also provided below for reference:

entity_name    metric              time         value  p95_threshold
80                ad  workload  1775313489000000  4.540000e+02   4.387500e+02
121        inventory       mem  1775313489000000  1.112381e+09   7.172106e+08
153               ad  workload  1775313519000000  5.480000e+02   4.387500e+02
157             cart       mem  1775313519000000  1.277542e+08   1.274808e+08
179   frontend-proxy   latency  1775313519000000  6.156100e-01   3.832555e-01
..               ...       ...               ...           ...            ...
777        inventory   latency  1775313759000000  1.227272e+01   1.112536e+01
782   load-generator       mem  1775313759000000  1.654557e+09   1.654444e+09
788  product-catalog   latency  1775313759000000  8.222250e-05   7.543801e-05
795   recommendation       cpu  1775313759000000  2.583109e-02   2.235157e-02
797   recommendation       mem  1775313759000000  2.282796e+08   2.282691e+08

[141 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t057/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure relevant columns are numeric for filtering and analysis
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces within the alert time window
traces_filtered = traces_data[
    (traces_data['startTime'] >= alert_start * 1_000) & (traces_data['endTime'] <= alert_end * 1_000)
]

# Identify traces where the 'cart' service is involved
cart_traces = traces_filtered[traces_filtered['serviceName'].str.contains('cart', case=False, na=False)]

# Check for errors or high latencies in the 'cart' service traces
cart_traces_anomalies = cart_traces[
    (cart_traces['statusCode'] != 0) | (cart_traces['duration'] > cart_traces['duration'].quantile(0.95))
]

# Extract downstream services in the call chain and their statuses
downstream_services = cart_traces_anomalies[['traceId', 'spanName', 'serviceName', 'statusCode', 'duration']]

downstream_services

```
Out[5]:
```
Within the alert time window, 3,355 traces involving the `cart` service were identified. Among these, some traces show high latencies or errors. For example, the `POST /oteldemo.CartService/AddItem` operation has traces with extremely high durations (e.g., over 76 seconds) and error status codes (`statusCode = 2`). Additionally, downstream operations like `GetInventory` and `GET` also exhibit errors and high latencies. These findings suggest potential issues in the `cart` service and its downstream dependencies. For a detailed view, additional rows can be displayed.

The original code execution output of IPython Kernel is also provided below for reference:

traceId                            spanName serviceName statusCode     duration
281329  aaa16df997df02c1b89a2f1ca14c1a73  POST /oteldemo.CartService/GetCart        cart          0      1751600
281330  67060e0458611da242ee2ee173d9e3ce                        GetInventory        cart          0      3185600
281331  67060e0458611da242ee2ee173d9e3ce  POST /oteldemo.CartService/AddItem        cart          0      6417100
281332  67060e0458611da242ee2ee173d9e3ce  POST /oteldemo.CartService/GetCart        cart          0      1724200
281333  0601375f08c0534d1fb63ada4781e8c9                        GetInventory        cart          0      2615400
...                                  ...                                 ...         ...        ...          ...
310004  3e769ee2f601c06a4342254696071f30  POST /oteldemo.CartService/AddItem        cart          2  76298220800
310005  3e769ee2f601c06a4342254696071f30                                 GET        cart          2  76294561300
310012  65c45ed51d8e378e53c989fb6ce3f7ec                                 GET        cart          2  76934562500
310013  65c45ed51d8e378e53c989fb6ce3f7ec                        GetInventory        cart          2  76935297400
310014  65c45ed51d8e378e53c989fb6ce3f7ec  POST /oteldemo.CartService/AddItem        cart          2  76937651600

[3355 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t057/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert the `_time_` column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs related to the `cart` service and its downstream services during the alert time window
cart_related_logs = logs_data[
    (logs_data['_time_'] >= alert_start) & (logs_data['_time_'] <= alert_end) &
    (logs_data['_container_name_'].str.contains('cart', case=False, na=False) |
     logs_data['_container_name_'].str.contains('inventory', case=False, na=False) |
     logs_data['_container_name_'].str.contains('frontend-proxy', case=False, na=False))
]

# Search for error messages or indications of resource issues
error_logs = cart_related_logs[
    cart_related_logs['content'].str.contains('error|memory|cpu|exception|fail|timeout', case=False, na=False)
]

error_logs

```
Out[6]:
```
No error messages or indications of resource issues, such as memory pressure or high CPU usage, were found in the logs related to the `cart` service and its downstream services during the alert time window.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```